# Notebook 03 - Ablation switches S2 and S3  (revision 2)
**Project:** What Makes AI-Based Sparse-View 3D Gaussian Splatting Work? - Farida Gaber & Mona Yasser

## Section 1 - Research overview
| Label | Config (runner name) | 3D points | Camera poses | Pose refinement | Compared with | Isolates |
|---|---|---|---|---|---|---|
| baseline | C3 | MASt3R | MASt3R | ON | - | - |
| **S2** | **C2** | MASt3R | MASt3R | **OFF** | C3 | contribution of pose refinement (RQ3 = C3 - C2) |
| **S3** | **C4** | **Random** | MASt3R | ON | C3 | contribution of MASt3R 3D points (RQ2 = C3 - C4) |

Do not confuse the labels: **S2 = C2**, **S3 = C4**.

**Re-uses Notebook 02.** `run_all.py`, `colab_prepare.py`, `pose_error.py`, `aggregate.py` are loaded from `Drive/scripts` (written by Notebook 02); nothing of the pipeline is redefined. This notebook adds the C4 hook, an execution planner, a strict pilot gate, per-run validation, aggregation and a matched comparison.

**What S3 really changes (verified in the InstantSplat source at commit `b951567`, see Section 7).** Only `sparse_<n>/0/points3D.ply` (point positions + colours + normals) and `confidence_dsp.npy` are replaced. The point *positions* decide (a) where the Gaussians start and (b) their initial size (`distCUDA2` nearest-neighbour distance). `confidence_dsp.npy` only scales the learning rate of the Gaussian positions (`xyz`). Everything else (poses, intrinsics, test cameras, training command except `--optim_pose` staying ON) is untouched.

**Before first use:** Notebooks 01 and 02 must each have been run once on this account. They do not need to be re-run for every session of Notebook 03.

## Section 3 - Configuration
- `MODE = "pilot"` is **strict**: C4, garden, 3 views, fixed seed, nothing else. It writes a *pilot gate* file only if every critical validation passes.
- `MODE = "full"` refuses to run unless that gate exists for the same seed, colour policy and InstantSplat commit (`ALLOW_FULL_WITHOUT_PILOT` exists only as an explicit, visible override).
- `DRY_RUN = True` (default) prints the audit, the plan and the commands and starts nothing; a CPU runtime is enough.
- `PAIRS_FROM_EXISTING_C3 = True` (full mode) takes the scene/view pairs from the valid C3 runs on Drive, so every C2/C4 run has an exactly matching C3.

In [19]:
DRIVE = "/content/drive/MyDrive/3dgs-project"
REPO, DATA, LOCAL_OUT = "/content/InstantSplat", "/content/data", "/content/out"
RES_DIR, TEST_STRIDE = "images_8", 8
ALL_SCENES = ["bicycle", "garden", "stump", "treehill", "bonsai", "counter", "kitchen", "room"]
OUTDOOR, INDOOR = ["bicycle", "garden", "stump", "treehill"], ["bonsai", "counter", "kitchen", "room"]

MODE = "pilot"                         # "pilot" (strict: C4 / garden / 3 views) or "full"
DRY_RUN = False                        # True = plan + commands only, nothing is started
RANDOM_SEED = 0                        # C4 seed (= the runner default Context.seed); keep it fixed across all runs
C4_COLOR_POLICY = "preserve_original"  # "preserve_original" (recommended, see Section 7) or "random" (sensitivity check only)
FORCE = False                          # True = allow re-running stale or already valid runs (old result.json is kept as .superseded_*)
STOP_ON_FAILURE = True                 # stop the batch at the first failed run or failed validation
ALLOW_FULL_WITHOUT_PILOT = False       # keep False: the full matrix must follow a passed pilot

# used only when MODE = "full"
CONFIGS_TO_RUN = ["C2", "C4"]          # S2 = C2, S3 = C4 ("C3" may be added; "C1" is not part of this notebook)
SCENES = ["garden", "bonsai"]
VIEWS = [3, 6, 9]
PAIRS_FROM_EXISTING_C3 = False         # True = use exactly the (scene, views) pairs that already have a valid C3 run

assert MODE in ("pilot", "full") and C4_COLOR_POLICY in ("preserve_original", "random")
assert isinstance(RANDOM_SEED, int)
if MODE == "pilot":
    CONFIGS_TO_RUN, SCENES, VIEWS, PAIRS_FROM_EXISTING_C3 = ["C4"], ["garden"], [3], False
    print("PILOT MODE (strict): C4 / garden / 3 views / seed", RANDOM_SEED)
assert set(CONFIGS_TO_RUN) <= {"C2", "C3", "C4"}, "Only C2 (S2), C3 (baseline) and C4 (S3) belong to this notebook."
assert set(SCENES) <= set(ALL_SCENES) and set(VIEWS) <= {3, 6, 9}, "unknown scene or view count"
print(dict(MODE=MODE, DRY_RUN=DRY_RUN, CONFIGS_TO_RUN=CONFIGS_TO_RUN, SCENES=SCENES, VIEWS=VIEWS, RANDOM_SEED=RANDOM_SEED,
           C4_COLOR_POLICY=C4_COLOR_POLICY, FORCE=FORCE, STOP_ON_FAILURE=STOP_ON_FAILURE))

PILOT MODE (strict): C4 / garden / 3 views / seed 0
{'MODE': 'pilot', 'DRY_RUN': False, 'CONFIGS_TO_RUN': ['C4'], 'SCENES': ['garden'], 'VIEWS': [3], 'RANDOM_SEED': 0, 'C4_COLOR_POLICY': 'preserve_original', 'FORCE': False, 'STOP_ON_FAILURE': True}


## Section 2 - Environment and Drive setup
Mounts Drive, loads the scripts from `Drive/scripts`, and (unless `DRY_RUN`) checks the GPU and restores the InstantSplat install from the Drive cache (1-3 min; nothing is rebuilt if the session is still alive). Applies the same two InstantSplat patches as Notebooks 01/02 (same markers, never twice) and checks the photo lists against `views_manifest.json`.

In [20]:
import os, sys, json, time, math, shutil, hashlib, datetime, importlib, subprocess, platform, tempfile
from pathlib import Path
import numpy as np
from google.colab import drive
drive.mount("/content/drive")
assert os.path.isdir(DRIVE), f"{DRIVE} not found - run Notebook 01 first (and add the shared-folder shortcut on Mona's account)."

REQUIRED = ["pose_error.py", "run_all.py", "aggregate.py", "colab_prepare.py", "recompute_pose_error.py"]
missing = [f for f in REQUIRED if not os.path.exists(f"{DRIVE}/scripts/{f}")]
if missing:
    raise SystemExit(f"Missing on Drive/scripts: {missing}. Run the first cell of Notebook 02 once (it writes them).")
os.makedirs("/content/scripts", exist_ok=True)
for f in REQUIRED:
    shutil.copy(f"{DRIVE}/scripts/{f}", f"/content/scripts/{f}")
sys.path.insert(0, "/content/scripts")
import pose_error, run_all, aggregate, colab_prepare, recompute_pose_error
for m in (pose_error, run_all, aggregate, colab_prepare, recompute_pose_error):
    importlib.reload(m)
pose_error.selftest()
aggregate.selftest()

if DRY_RUN:
    GPU_NAME, COMMIT = "CPU (dry run)", colab_prepare.INSTANTSPLAT_COMMIT
else:
    GPU_NAME = colab_prepare.check_gpu()
    COMMIT = colab_prepare.install(DRIVE, REPO)
counts = colab_prepare.restore_data(DRIVE, DATA, ALL_SCENES, RES_DIR)
if not DRY_RUN:
    colab_prepare.patch_instantsplat(REPO)
manifest = colab_prepare.build_manifest(DATA, ALL_SCENES, [3, 6, 9], RES_DIR, TEST_STRIDE, DRIVE)

ctx = run_all.Context(drive=DRIVE, repo=REPO, data=DATA, local_out=LOCAL_OUT, manifest=manifest,
                      commit=COMMIT, gpu=GPU_NAME, res_dir=RES_DIR, test_stride=TEST_STRIDE, seed=RANDOM_SEED)
ctx.c4_color_policy = C4_COLOR_POLICY          # read by the C4 hook
RESULTS = f"{DRIVE}/results"
os.makedirs(f"{DRIVE}/logs", exist_ok=True)
SESSION = datetime.datetime.now().strftime("%Y-%m-%d_%H%M")
print("GPU:", GPU_NAME, "| InstantSplat:", COMMIT, "| photos per scene:", counts)

Drive already mounted at /content/drive; to attempt to forcibly remount, call drive.mount("/content/drive", force_remount=True).
selftest OK - identity, Sim(3)-invariance, w2c round trip, a known 10-degree error and near-collinear 3-camera stability all pass
11 runs -> all_runs.csv, means_by_config.csv, effects.csv in /tmp/tmpbhr251i4
  C1 3 photos (2 scenes): PSNR 16.50  ATE 0.000  rot 0.00 deg
  C2 3 photos (3 scenes): PSNR 13.67  ATE 0.100  rot 1.00 deg
  C3 3 photos (3 scenes): PSNR 14.33  ATE 0.100  rot 1.00 deg
  C4 3 photos (3 scenes): PSNR 12.33  ATE 0.100  rot 1.00 deg
  RQ1 poses (C1 - C2) at 3 photos (2 scenes): dPSNR +2.00, sd 0.00, positive in 2/2 scenes
  RQ2 3D points (C3 - C4) at 3 photos (2 scenes): dPSNR +1.50, sd 0.00, positive in 2/2 scenes
  RQ3 refinement (C3 - C2) at 3 photos (3 scenes): dPSNR +0.67, sd 0.29, positive in 3/3 scenes
selftest OK - aggregate ignores old/baseline folders and compares only matching scenes
cached wheels OK
setup OK 2.14.1+cu130

patche

## Section 4 - Existing results and experiment audit
Classifies every requested run:
- `done` - valid result whose parameters match this session (scene, views, config, iterations, resolution, test-pose iterations, stride, commit, photo list; for C4 also seed, colour policy and a passed validation);
- `stale` - a result exists but a parameter differs. It is **not reused**; it is only re-run with `FORCE = True` (the old file is kept as `result.superseded_<time>.json`);
- `failed` - the run folder has files but no valid `result.json`; retried automatically, logs kept;
- `missing` - never run.

It also lists the **matched-pair coverage**: for every (scene, views) that already has a valid C3 run, whether the matching C2 and C4 runs exist. Only exact (scene, views) matches are ever compared (Section 14).

In [21]:
def _alias_dirs(c, s, v):
    out = [Path(run_all.result_dir(ctx, c, s, v))]
    if c in ctx.reuse:                                   # C3 -> C3_baseline (same settings, run by Notebook 01)
        out.append(Path(run_all.result_dir(ctx, ctx.reuse[c], s, v)))
    return out


def _mismatches(r, c, s, v):
    exp = dict(iterations=ctx.iters, resolution=ctx.res_dir, optim_test_pose_iter=ctx.optim_test_pose_iter, test_stride=ctx.test_stride)
    if not DRY_RUN:
        exp["instantsplat_commit"] = COMMIT
    bad = [f"{k}: result={r[k]!r} vs now={e!r}" for k, e in exp.items() if k in r and r[k] != e]
    allowed = {c} | ({"C3_baseline"} if c == "C3" else set())
    if r.get("config") not in allowed:
        bad.append(f"config: result={r.get('config')!r} vs {c!r}")
    if r.get("scene") != s or r.get("n_views") != v:
        bad.append("scene/views differ from folder")
    if r.get("pose_metric") != "pose_error.py":
        bad.append("pose_metric is not pose_error.py")
    if r.get("train_images") not in (None, manifest[s]["train_images"][str(v)]):
        bad.append("training photo list differs from manifest")
    if c == "C4":
        si = r.get("switch_info") or {}
        if r.get("seed") != RANDOM_SEED:
            bad.append(f"seed: result={r.get('seed')!r} vs now={RANDOM_SEED!r}")
        if si.get("color_policy") != C4_COLOR_POLICY:
            bad.append(f"color_policy: result={si.get('color_policy')!r} vs now={C4_COLOR_POLICY!r}")
        if si.get("validation") != "PASSED":
            bad.append("C4 switch_info has no PASSED validation")
    return bad


def classify(c, s, v):
    # returns (status, detail)
    for d in _alias_dirs(c, s, v):
        f = d / "result.json"
        if f.exists():
            try:
                r = json.load(open(f))
            except Exception as e:
                return "failed", f"unreadable result.json ({e})"
            bad = _mismatches(r, c, s, v)
            return ("done", str(d.parent.parent.name)) if not bad else ("stale", "; ".join(bad))
    d = _alias_dirs(c, s, v)[0]
    if d.exists() and any(d.iterdir()):
        return "failed", "run folder has files but no result.json (logs kept)"
    return "missing", ""


rows_existing = aggregate.load(RESULTS)
print(f"{len(rows_existing)} valid run(s) found in {RESULTS}:")
for r in rows_existing:
    print(f"  {r['config']:3s} {r['scene']:9s} {r['n_views']} views  PSNR {r['psnr']:.3f}  ATE {r['ate']:.4f}  rot {r['rot_deg']:.3f}")

# ---- matched-pair coverage: every valid C3 run needs its C2 and C4 counterpart
C3_PAIRS = sorted({(r["scene"], r["n_views"]) for r in rows_existing if r["config"] == "C3"})
COVERAGE = {(s, v): dict(C2=classify("C2", s, v)[0], C4=classify("C4", s, v)[0]) for s, v in C3_PAIRS}
print("\nMatched-pair coverage (rows = existing valid C3 runs):")
for (s, v), cv in COVERAGE.items():
    print(f"  {s:9s} {v} views : C3 done | C2 {cv['C2']:8s} | C4 {cv['C4']:8s}")
MISSING_MATCHED_C2 = [(s, v) for (s, v), cv in COVERAGE.items() if cv["C2"] != "done"]
MISSING_MATCHED_C4 = [(s, v) for (s, v), cv in COVERAGE.items() if cv["C4"] != "done"]
print(f"\nMISSING matched C2 (S2) runs ({len(MISSING_MATCHED_C2)}):", MISSING_MATCHED_C2)
print(f"MISSING matched C4 (S3) runs ({len(MISSING_MATCHED_C4)}):", MISSING_MATCHED_C4)

# ---- (scene, views) pairs of this session
if MODE == "pilot":
    PAIRS = [("garden", 3)]
elif PAIRS_FROM_EXISTING_C3:
    PAIRS = list(C3_PAIRS)
else:
    PAIRS = [(s, v) for s in SCENES for v in VIEWS]
AUDIT = {(c, s, v): classify(c, s, v) for c in sorted(set(CONFIGS_TO_RUN) | {"C3"}) for (s, v) in PAIRS}
print("\nStatus of this session's runs (C3 shown as matched baseline):")
for (c, s, v), (st, det) in AUDIT.items():
    print(f"  {c} {s:9s} {v} views : {st:8s} {det}")

7 valid run(s) found in /content/drive/MyDrive/3dgs-project/results:
  C2  garden    3 views  PSNR 14.619  ATE 0.0193  rot 1.092
  C3  bonsai    3 views  PSNR 14.706  ATE 0.1066  rot 6.855
  C3  bonsai    6 views  PSNR 15.305  ATE 0.0785  rot 4.581
  C3  bonsai    9 views  PSNR 16.537  ATE 0.1379  rot 2.228
  C3  garden    3 views  PSNR 14.710  ATE 0.0182  rot 1.109
  C3  garden    6 views  PSNR 17.781  ATE 0.0259  rot 1.320
  C3  garden    9 views  PSNR 18.151  ATE 0.0192  rot 0.750

Matched-pair coverage (rows = existing valid C3 runs):
  bonsai    3 views : C3 done | C2 missing  | C4 missing 
  bonsai    6 views : C3 done | C2 missing  | C4 missing 
  bonsai    9 views : C3 done | C2 missing  | C4 missing 
  garden    3 views : C3 done | C2 done     | C4 missing 
  garden    6 views : C3 done | C2 missing  | C4 missing 
  garden    9 views : C3 done | C2 missing  | C4 missing 

MISSING matched C2 (S2) runs (5): [('bonsai', 3), ('bonsai', 6), ('bonsai', 9), ('garden', 6), ('garden', 

## Section 5 - Initialization and data validation
Contract used by the hook, **taken from the InstantSplat source at `b951567`** (not assumed):
- `init_geo.py -> save_points3D` writes `sparse_<n>/0/points3D.ply` through `storePly`: vertex properties `x,y,z,nx,ny,nz` (float32) and `red,green,blue` (uint8), normals all zero, colours = image pixel colours; it writes `confidence_dsp.npy` with `confs.reshape(-1, 1)`, i.e. shape **(N, 1)**, one row per vertex.
- Cameras: `save_extrinsic` writes **world-to-camera** `qvec`/`tvec` to `images.txt` and `images.bin`; the camera centre is `-R^T t`.
- Training reads the PLY through `fetchPly` and the confidence through `load_and_prepare_confidence` (`1 - sigmoid(conf)`, scaled to 1..100, used as per-point learning-rate factor for `xyz`).

`preflight_init` checks every item of this contract on the real files and **stops without modifying anything** if one is violated. It runs (a) here on every existing MASt3R cache (extracted to a temporary folder, the Drive cache is only read) and (b) automatically inside the C4 hook before any change. If no cache exists yet, the first run creates it; the hook's preflight then runs on the fresh output before the PLY is touched.

In [22]:
EXPECTED_PLY = [("x", "f4"), ("y", "f4"), ("z", "f4"), ("nx", "f4"), ("ny", "f4"), ("nz", "f4"),
                ("red", "u1"), ("green", "u1"), ("blue", "u1")]          # storePly() in scene/dataset_readers.py @ b951567


def _sha(path):
    h = hashlib.sha256()
    with open(path, "rb") as f:
        for b in iter(lambda: f.read(1 << 20), b""):
            h.update(b)
    return h.hexdigest()


def _atomic_json(path, obj):
    tmp = f"{path}.tmp"
    with open(tmp, "w") as f:
        json.dump(obj, f, indent=1, default=str)
    os.replace(tmp, path)


def _qvec_to_R(q):
    w, x, y, z = np.asarray(q, float) / np.linalg.norm(q)
    return np.array([[1 - 2 * (y * y + z * z), 2 * (x * y - w * z), 2 * (x * z + w * y)],
                     [2 * (x * y + w * z), 1 - 2 * (x * x + z * z), 2 * (y * z - w * x)],
                     [2 * (x * z - w * y), 2 * (y * z + w * x), 1 - 2 * (x * x + y * y)]])


def _parse_images_txt(path):
    # COLMAP images.txt as written by save_extrinsic: IMAGE_ID QW QX QY QZ TX TY TZ CAMERA_ID NAME (+ empty point line)
    out = []
    for line in open(path):
        p = line.split()
        if len(p) == 10 and not line.startswith("#"):
            out.append((int(p[0]), [float(x) for x in p[1:5]], [float(x) for x in p[5:8]], p[9]))
    return sorted(out)


def _camera_centres(images_txt):
    rows = _parse_images_txt(images_txt)
    return np.array([-_qvec_to_R(q).T @ np.array(t) for _, q, t, _ in rows], dtype=np.float64), [r[3] for r in rows]


def _snapshot(sp, exclude=()):
    return {str(p.relative_to(sp)): _sha(p) for p in sorted(Path(sp).rglob("*")) if p.is_file() and p.name not in exclude}


def _fetch_ply_replica(path):
    # exact copy of fetchPly() in scene/dataset_readers.py @ b951567 (returns plain arrays)
    from plyfile import PlyData
    v = PlyData.read(path)["vertex"]
    pos = np.vstack([v["x"], v["y"], v["z"]]).T
    col = np.vstack([v["red"], v["green"], v["blue"]]).T / 255.0
    nor = np.vstack([v["nx"], v["ny"], v["nz"]]).T
    return pos, col, nor


def _conf_lr_modifiers(conf):
    # numpy replica of load_and_prepare_confidence(..., scale=(1, 100)) in train.py @ b951567
    sig = 1.0 / (1.0 + np.exp(-np.asarray(conf, dtype=np.float64)))
    return (1.0 - sig) * (100 - 1) + 1


def preflight_init(sp, n_views, expected_names=None, tag="preflight"):
    # Validates the real MASt3R output against the loader contract. Raises (and changes nothing) on any violation.
    from plyfile import PlyData
    sp = Path(sp)
    need = ["0/points3D.ply", "0/confidence_dsp.npy", "0/images.txt", "0/images.bin", "0/cameras.txt", "1/images.txt"]
    miss = [f for f in need if not (sp / f).exists()]
    if miss:
        raise FileNotFoundError(f"{tag}: missing in {sp}: {miss}")
    ply = PlyData.read(str(sp / "0/points3D.ply"))
    if [e.name for e in ply.elements] != ["vertex"]:
        raise ValueError(f"{tag}: PLY elements {[e.name for e in ply.elements]} != ['vertex']")
    arr = ply["vertex"].data
    got = [(n, arr.dtype[n].kind + str(arr.dtype[n].itemsize)) for n in arr.dtype.names]
    if got != EXPECTED_PLY:
        raise ValueError(f"{tag}: PLY layout {got} differs from InstantSplat storePly layout {EXPECTED_PLY}")
    n = len(arr)
    if n == 0:
        raise ValueError(f"{tag}: empty point cloud")
    xyz = np.stack([arr["x"], arr["y"], arr["z"]], 1).astype(np.float64)
    finite = np.isfinite(xyz).all(1)
    if not finite.any():
        raise ValueError(f"{tag}: no finite points")
    conf = np.load(sp / "0/confidence_dsp.npy")
    if conf.shape != (n, 1) or conf.dtype.kind != "f":
        raise ValueError(f"{tag}: confidence_dsp.npy has shape {conf.shape} dtype {conf.dtype}; contract is ({n}, 1) float")
    if not np.isfinite(conf).any():
        raise ValueError(f"{tag}: confidence has no finite value")
    cams, names = _camera_centres(sp / "0/images.txt")
    if len(cams) != n_views or not np.isfinite(cams).all():
        raise ValueError(f"{tag}: expected {n_views} finite training cameras in images.txt, found {len(cams)}")
    if expected_names is not None and names != list(expected_names):
        raise ValueError(f"{tag}: camera names {names} != manifest {list(expected_names)}")
    n_test = len(_parse_images_txt(sp / "1/images.txt"))
    ev = dict(ply_path=str(sp / "0/points3D.ply"), n_vertices=n, n_finite=int(finite.sum()),
              ply_layout=got, normals_abs_max=float(max(np.abs(arr[k]).max() for k in ("nx", "ny", "nz"))),
              colour_min=int(min(arr[k].min() for k in ("red", "green", "blue"))), colour_max=int(max(arr[k].max() for k in ("red", "green", "blue"))),
              confidence=dict(path=str(sp / "0/confidence_dsp.npy"), shape=list(conf.shape), dtype=str(conf.dtype),
                              mean=float(conf[np.isfinite(conf)].mean()), min=float(np.nanmin(conf)), max=float(np.nanmax(conf)),
                              all_finite=bool(np.isfinite(conf).all())),
              camera_names=names, camera_centres_method="-R^T t from world-to-camera qvec/tvec in images.txt (save_extrinsic)",
              n_test_cameras=n_test, bbox_points_min=xyz[finite].min(0).tolist(), bbox_points_max=xyz[finite].max(0).tolist(),
              camera_centres_min=cams.min(0).tolist(), camera_centres_max=cams.max(0).tolist())
    try:
        import pose_error as _pe
        b = _pe.read_images_bin(str(sp / "0/images.bin"))
        c_bin = np.array([_pe.w2c_to_cam(*b[nm])[1] for nm in names])
        diff = float(np.abs(c_bin - cams).max())
        ev["centres_txt_vs_bin_max_abs_diff"] = diff
        if diff > 1e-3 * max(1.0, float(np.abs(cams).max())):
            raise ValueError(f"{tag}: camera centres from images.txt and images.bin disagree ({diff:.3e})")
    except ImportError:
        ev["centres_txt_vs_bin_max_abs_diff"] = None
    return ev


def source_contract_report():
    # compares the installed InstantSplat with the contract above; stops if storePly changed
    if not os.path.isdir(REPO):
        print("(InstantSplat is not installed in this dry-run session: contract verified by the notebook author on GitHub source b951567, not re-verified here)")
        return None
    rd = Path(f"{REPO}/scene/dataset_readers.py").read_text()
    ok = all(f"('{n}', '{t}')" in rd for n, t in EXPECTED_PLY)
    cons = subprocess.run(f"cd {REPO} && grep -rn --include=*.py -E 'points3D_all|pointsColor_all|confidence.npy|confidence_dsp' . | grep -v submodules",
                          shell=True, capture_output=True, text=True).stdout
    print("storePly layout in installed repo matches the contract:", ok)
    print("consumers of the MASt3R side files in the installed repo:\n" + cons)
    if not ok:
        raise SystemExit("Installed InstantSplat storePly differs from the contract the C4 hook relies on - stopping.")
    return cons


def check_env():
    for s in sorted({s for s, _ in PAIRS}):
        for p in [f"{DATA}/{s}/{RES_DIR}", f"{DATA}/{s}/sparse/0/images.bin"]:
            assert os.path.exists(p), f"MISSING: {p}"
        assert len(os.listdir(f"{DATA}/{s}/{RES_DIR}")) == manifest[s]["n_images"], f"{s}: image count differs from manifest"
    for s, v in PAIRS:
        tr = manifest[s]["train_images"][str(v)]
        assert len(tr) == v and len(set(tr)) == v and not set(tr) & set(manifest[s]["test_images"]), f"{s} {v}: bad split"
    ck = f"{DRIVE}/checkpoints/MASt3R_ViTLarge_BaseDecoder_512_catmlpdpt_metric.pth"
    print("MASt3R checkpoint on Drive:", "OK" if os.path.exists(ck) else "MISSING (needed only for runs without init cache)")
    for d in [RESULTS, f"{DRIVE}/logs", f"{DRIVE}/init_cache"]:
        os.makedirs(d, exist_ok=True)
        t = f"{d}/.w_{os.getpid()}"; open(t, "w").write("x"); os.remove(t)
    if not DRY_RUN:
        import diff_gaussian_rasterization, simple_knn, fused_ssim
    try:
        import plyfile
    except ImportError:
        print("Installing missing plyfile package...")
        subprocess.run([sys.executable, "-m", "pip", "install", "plyfile"], check=True)
        import plyfile
    return True


def preflight_cached(s, v):
    d = run_all.init_cache_dir(ctx, s, v)
    if not os.path.exists(f"{d}/sparse.tar"):
        return None
    tmp = tempfile.mkdtemp(prefix="inspect_")
    try:
        subprocess.run(["tar", "-xf", f"{d}/sparse.tar", "-C", tmp], check=True)
        return preflight_init(Path(tmp) / f"sparse_{v}", v, manifest[s]["train_images"][str(v)], tag=f"[{s} {v} views | cache preflight]")
    finally:
        shutil.rmtree(tmp, ignore_errors=True)


print("environment ok:", check_env())
source_contract_report()
PREFLIGHT = {}
for s, v in PAIRS:
    ev = preflight_cached(s, v)
    PREFLIGHT[(s, v)] = ev
    print(f"\n{s}/{v}_views MASt3R cache:", "not present yet - the first run creates it; the hook preflight then runs before any change" if ev is None else "PREFLIGHT PASSED")
    if ev:
        for k, val in ev.items():
            print(f"   {k}: {val}")

MASt3R checkpoint on Drive: OK
environment ok: True
storePly layout in installed repo matches the contract: True
consumers of the MASt3R side files in the installed repo:
./train.py:95:    confidence_path = os.path.join(dataset.source_path, f"sparse_{dataset.n_views}/0", "confidence_dsp.npy")
./init_test_pose.py:48:    train_pts_all_path = sparse_0_path / 'points3D_all.npy'
./utils/sfm_utils.py:260:        np.save(sparse_path / 'confidence.npy', confs)
./utils/sfm_utils.py:293:        np.save(sparse_path / 'confidence_dsp.npy', confs)
./utils/sfm_utils.py:297:        np.save(sparse_path / 'points3D_all.npy', pts3d)
./utils/sfm_utils.py:298:        np.save(sparse_path / 'pointsColor_all.npy', imgs)


garden/3_views MASt3R cache: PREFLIGHT PASSED
   ply_path: /tmp/inspect_icecjccv/sparse_3/0/points3D.ply
   n_vertices: 486815
   n_finite: 486815
   ply_layout: [('x', 'f4'), ('y', 'f4'), ('z', 'f4'), ('nx', 'f4'), ('ny', 'f4'), ('nz', 'f4'), ('red', 'u1'), ('green', 'u1'), ('blue', 'u1')]

## Section 6 - S2 implementation (C2: MASt3R init, refinement OFF)
`run_all.CONFIGS["C2"]` is `after_init=hook_none, optim_pose=False`; `run_all.commands()` omits `--optim_pose` for it, so S2 needs no new training code.

**How pose optimization is switched off in InstantSplat (`train.py` @ b951567):** every iteration does `if opt.optim_pose == False: gaussians.P.requires_grad_(False)`; `--optim_pose` defaults to `False` in `arguments/__init__.py`. The pose tensor `P` therefore receives no gradient and the Adam step leaves it untouched. This is **not** a zero learning rate; it is a stricter off-switch (no gradient at all). The notebook does not touch the optimizer or any learning rate.

Two **independent** pose checks are used, both implemented in `pose_stage_checks`:
1. *Replacement stage (C4 only)*: all camera/pose files of `sparse_<n>` are byte-identical before and after the hook (hash comparison, Section 7).
2. *Training stage*: **C2** must end with exactly the starting poses (`max|pose_optimized - pose_org| < 1e-6`, ATE and rotation error unchanged); **C3/C4** must have the `--optim_pose` flag in the training command and their poses must have *moved* (evidence that refinement is active). C3/C4 are never required to stay unchanged.

In [23]:
POSE_TOL = 1e-6

def _cmd(config, scene="garden", v=3):
    cfg = run_all.CONFIGS[config]
    return dict(run_all.commands(ctx, cfg, f"{DATA}/{scene}", f"{LOCAL_OUT}/{config}/{scene}/{v}_views", v))

c2, c3, c4 = _cmd("C2"), _cmd("C3"), _cmd("C4")
assert "--optim_pose" not in c2["train"], "C2 must NOT pass --optim_pose"
assert "--optim_pose" in c3["train"] and "--optim_pose" in c4["train"], "C3/C4 must pass --optim_pose"
norm = lambda d: {k: val.replace(" --optim_pose", "").replace("/C2/", "/X/").replace("/C3/", "/X/").replace("/C4/", "/X/") for k, val in d.items()}
assert norm(c2) == norm(c3) == norm(c4), "commands differ in something other than --optim_pose"
assert run_all.CONFIGS["C2"]["optim_pose"] is False and run_all.CONFIGS["C3"]["optim_pose"] is True and run_all.CONFIGS["C4"]["optim_pose"] is True
print("train C2:", c2["train"]); print("train C4:", c4["train"])
print("OK - C2 omits --optim_pose; C3/C4 pass it; every other argument is identical.")

if os.path.isdir(REPO):
    print("\nInstalled InstantSplat, lines that gate pose optimization:")
    print(subprocess.run(f"cd {REPO} && grep -n -E 'optim_pose|requires_grad_\\(False\\)' train.py arguments/__init__.py", shell=True, capture_output=True, text=True).stdout)


def pose_stage_checks(config, scene, v):
    # training-stage pose behaviour of a finished run (replacement-stage preservation is checked separately for C4)
    d = Path(run_all.result_dir(ctx, config, scene, v))
    if not (d / "poses/pose_org.npy").exists():
        d = Path(run_all.result_dir(ctx, ctx.reuse.get(config, config), scene, v))
    p0, p1 = np.load(d / "poses/pose_org.npy"), np.load(d / "poses/pose_optimized.npy")
    assert p0.shape == p1.shape and np.isfinite(p0).all() and np.isfinite(p1).all(), "pose arrays malformed"
    moved = float(np.abs(p1 - p0).max())
    r = json.load(open(d / "result.json"))
    d_ate, d_rot = abs(r["ate"] - r["ate_init"]), abs(r["rot_deg"] - r["rot_deg_init"])
    train_cmd = _cmd(config, scene, v)["train"]
    info = dict(max_abs_pose_difference=moved, abs_ate_change=d_ate, abs_rot_change_deg=d_rot)
    if config == "C2":
        checks = dict(train_command_omits_optim_pose="--optim_pose" not in train_cmd,
                      poses_unchanged_after_training=moved < POSE_TOL, ate_unchanged=d_ate < POSE_TOL, rotation_unchanged=d_rot < 1e-5)
    else:
        checks = dict(train_command_enables_optim_pose="--optim_pose" in train_cmd, poses_changed_by_training=moved > POSE_TOL)
    return info, checks

if (Path(run_all.result_dir(ctx, "C2", "garden", 3)) / "poses/pose_org.npy").exists():
    print("existing C2 garden/3 (not re-run):", pose_stage_checks("C2", "garden", 3))

train C2: python train.py -s /content/data/garden -m /content/out/C2/garden/3_views -r 1 --n_views 3 --iterations 1000 --pp_optimizer
train C4: python train.py -s /content/data/garden -m /content/out/C4/garden/3_views -r 1 --n_views 3 --iterations 1000 --pp_optimizer --optim_pose
OK - C2 omits --optim_pose; C3/C4 pass it; every other argument is identical.

Installed InstantSplat, lines that gate pose optimization:
train.py:144:        if opt.optim_pose==False:
train.py:145:            gaussians.P.requires_grad_(False)
arguments/__init__.py:93:        self.optim_pose = False

existing C2 garden/3 (not re-run): ({'max_abs_pose_difference': 0.0, 'abs_ate_change': 0.0, 'abs_rot_change_deg': 0.0}, {'train_command_omits_optim_pose': True, 'poses_unchanged_after_training': True, 'ate_unchanged': True, 'rotation_unchanged': True})


## Section 7 - S3 implementation (C4: random 3D points, MASt3R poses, refinement ON)

### Point-attribute strategy (decided from the InstantSplat source, `b951567`)
| Attribute | How InstantSplat consumes it | What the hook does | Why |
|---|---|---|---|
| **x, y, z** | `create_from_pcd`: Gaussian centres; also `distCUDA2` nearest-neighbour distance -> initial Gaussian scale (`init_scale_from_view_depth` is `False` by default) | **replaced** by uniform random points | this is the variable under test; the induced initial scale is a consequence of the positions and is part of S3 |
| **nx, ny, nz** | `fetchPly` reads them, `create_from_pcd` never uses them; `storePly` writes zeros | kept (zeros, identical to the original) | not consumed -> not a variable |
| **red, green, blue** | `RGB2SH(colors/255)` -> initial `_features_dc` (trainable) | **default `preserve_original`**: the original per-point colour array is kept unchanged | keeps the initial colour *distribution* identical to C3, so only positions (and the colour-position correspondence they carry) change. `random` colours would additionally destroy the colour prior = a second variable |
| **confidence_dsp.npy** (N,1) | `1 - sigmoid(conf)` scaled to 1..100 = per-point learning-rate factor of the Gaussian **positions** only | constant = mean of the finite original values, same shape/dtype (as specified in Notebook 02) | random points have no MASt3R confidence; the original values belong to other locations |

**Interpretation of S3.** The comparison C3 - C4 measures the value of *MASt3R point information*: positions (with the initial scales they imply), the colour-to-position correspondence and the confidence-based position learning rates. It does **not** measure positions alone: the constant confidence is a second, unavoidable change (it is tied to the replaced points). Random colours (`C4_COLOR_POLICY = "random"`) are available only as a sensitivity check; the policy is stored in every result and a result with another policy counts as *stale*. This departs from the wording "random colours" in Notebook 02 on purpose, and is documented here.

`use_random_points` runs after the MASt3R output was saved to the Drive cache, on the local copy `DATA/<scene>/sparse_<n>` only. It: runs `preflight_init` (stops on any contract violation); computes the bounding box of all finite MASt3R points **and** the training-camera centres; draws exactly N = number of PLY vertices points uniformly per axis with `default_rng(ctx.seed)`, strictly inside the box in float32; replaces the PLY and the confidence atomically (originals are backed up to `model/random_points_backup/`); then verifies the result with the real `fetchPly` (in a subprocess from the installed repo; otherwise with the exact replica) and a before/after hash comparison of every other file in `sparse_<n>`. `random_points_meta.json` is always written (PASSED or FAILED).

In [24]:
def _real_loader_stats(repo, ply_path):
    # runs InstantSplat's own fetchPly in a subprocess; returns None if the repo is not usable here
    if not os.path.isdir(repo):
        return None
    code = ("import sys,json,numpy as np; sys.path.insert(0,%r); from scene.dataset_readers import fetchPly; p=fetchPly(%r); "
            "print('JSON:'+json.dumps(dict(n=int(len(p.points)),finite=bool(np.isfinite(p.points).all()),"
            "cmin=float(p.colors.min()),cmax=float(p.colors.max()),nabs=float(abs(p.normals).max()),"
            "psum=float(np.asarray(p.points,float).sum()))))") % (repo, str(ply_path))
    r = subprocess.run([sys.executable, "-c", code], cwd=repo, capture_output=True, text=True, timeout=600)
    if r.returncode != 0:
        return None
    line = [l for l in r.stdout.splitlines() if l.startswith("JSON:")]
    return json.loads(line[-1][5:]) if line else None


def use_random_points(ctx, scene, n_views, model):
    # C4 / S3 switch (see the Section 7 text for the attribute strategy)
    from plyfile import PlyData, PlyElement
    tag = f"[C4 {scene} {n_views} views | random_points]"
    policy = getattr(ctx, "c4_color_policy", "preserve_original")
    if policy not in ("preserve_original", "random"):
        raise ValueError(f"{tag} unknown colour policy {policy!r}")
    sp = Path(ctx.data) / scene / f"sparse_{n_views}"
    ply_path, conf_path = sp / "0" / "points3D.ply", sp / "0" / "confidence_dsp.npy"
    meta_path = f"{ctx.drive}/results/C4/{scene}/{n_views}_views/random_points_meta.json"
    os.makedirs(os.path.dirname(meta_path), exist_ok=True)
    meta = dict(scene=scene, n_views=n_views, config="C4", seed=int(ctx.seed), color_policy=policy, validation="FAILED",
                ply_filename=ply_path.name, ply_path=str(ply_path), confidence_path=str(conf_path))
    try:
        ev = preflight_init(sp, n_views, ctx.manifest[scene]["train_images"][str(n_views)], tag=tag + " preflight")
        meta["preflight"] = ev                                   # real layout, count, shapes, cameras of THIS run
        EXCL = {"points3D.ply", "confidence_dsp.npy"}
        before = _snapshot(sp, EXCL)
        arr = PlyData.read(str(ply_path))["vertex"].data
        n = len(arr)
        xyz = np.stack([arr["x"], arr["y"], arr["z"]], 1).astype(np.float64)
        finite = np.isfinite(xyz).all(1)
        cams, _ = _camera_centres(sp / "0" / "images.txt")
        allp = np.concatenate([xyz[finite], cams], 0)
        bmin, bmax = allp.min(0), allp.max(0)
        if not (np.isfinite(bmin).all() and np.isfinite(bmax).all() and (bmax > bmin).all()):
            raise ValueError(f"{tag} degenerate bounding box {bmin} {bmax}")
        lo = np.nextafter(bmin.astype(np.float32), np.float32(np.inf)).astype(np.float64)    # float32 samples stay inside the float64 box
        hi = np.nextafter(bmax.astype(np.float32), np.float32(-np.inf)).astype(np.float64)
        rng = np.random.default_rng(int(ctx.seed))
        rnd = rng.uniform(lo, hi, size=(n, 3)).astype(np.float32)
        new = arr.copy()
        new["x"], new["y"], new["z"] = rnd[:, 0], rnd[:, 1], rnd[:, 2]
        if policy == "random":                                   # sensitivity variant only
            for k in ("red", "green", "blue"):
                new[k] = rng.integers(0, 256, n).astype(np.uint8)
        conf = np.load(conf_path)
        mean_conf = float(conf[np.isfinite(conf)].mean())
        new_conf = np.full(conf.shape, mean_conf, dtype=conf.dtype)

        bdir = Path(model) / "random_points_backup"
        bdir.mkdir(parents=True, exist_ok=True)
        shutil.copy(ply_path, bdir / "points3D.mast3r.ply"); shutil.copy(conf_path, bdir / "confidence_dsp.mast3r.npy")
        h_ply0 = _sha(ply_path)
        tmp_ply, tmp_cf = sp / "0" / "points3D.ply.tmp", sp / "0" / "confidence_dsp.tmp.npy"
        PlyData([PlyElement.describe(new, "vertex")], text=False).write(str(tmp_ply))
        np.save(tmp_cf, new_conf)
        os.replace(tmp_ply, ply_path); os.replace(tmp_cf, conf_path)

        # ---- verification of what is now on disk, through the loader InstantSplat uses
        pos, col, nor = _fetch_ply_replica(str(ply_path))
        real = _real_loader_stats(ctx.repo, ply_path)
        chk = PlyData.read(str(ply_path))["vertex"].data
        cf2 = np.load(conf_path)
        mod_orig, mod_new = _conf_lr_modifiers(conf), _conf_lr_modifiers(cf2)
        checks = dict(
            same_point_count=len(chk) == n and pos.shape == (n, 3),
            same_layout=[(k, chk.dtype[k].kind + str(chk.dtype[k].itemsize)) for k in chk.dtype.names] == EXPECTED_PLY,
            coordinates_finite=bool(np.isfinite(pos).all()),
            inside_bbox=bool(((pos >= bmin) & (pos <= bmax)).all()),
            points_replaced=not np.array_equal(pos[finite], xyz[finite]),
            normals_as_original=bool(np.array_equal(nor, np.stack([arr["nx"], arr["ny"], arr["nz"]], 1))),
            colours_as_policy=(bool(np.array_equal(chk["red"], arr["red"]) and np.array_equal(chk["green"], arr["green"]) and np.array_equal(chk["blue"], arr["blue"]))
                               if policy == "preserve_original" else bool(((col >= 0) & (col <= 1)).all())),
            confidence_len=cf2.shape == (n, 1), confidence_dtype=cf2.dtype == conf.dtype, confidence_finite=bool(np.isfinite(cf2).all()),
            confidence_constant=bool(np.all(cf2 == cf2.flat[0])), lr_modifier_in_range=bool(((mod_new >= 1) & (mod_new <= 100)).all()),
            camera_and_other_files_unchanged=_snapshot(sp, EXCL) == before,
            real_or_replica_loader_agrees=(real is None) or (real["n"] == n and real["finite"] and abs(real["psum"] - float(pos.astype(np.float64).sum())) < 1e-3 * max(1.0, abs(real["psum"]))),
            original_ply_backed_up=_sha(bdir / "points3D.mast3r.ply") == h_ply0 and _sha(ply_path) != h_ply0)
        meta.update(n_original_points=n, n_finite_original_points=int(finite.sum()), n_random_points=int(len(chk)),
                    bbox_min=bmin.tolist(), bbox_max=bmax.tolist(), bbox_includes="all finite MASt3R points + training camera centres",
                    n_camera_centres=int(len(cams)), mean_confidence_used=mean_conf, confidence="constant (mean of finite original values)",
                    confidence_shape=list(conf.shape), confidence_dtype=str(conf.dtype),
                    lr_modifier_original_mean=float(mod_orig.mean()), lr_modifier_constant=float(mod_new.flat[0]),
                    loader_used="InstantSplat fetchPly (subprocess)" if real else "exact replica of fetchPly",
                    unconsumed_mast3r_side_files="confidence.npy, points3D_all.npy, pointsColor_all.npy remain (MASt3R, unchanged): train/render/metrics do not read them",
                    checks=checks)
        bad = [k for k, ok in checks.items() if not ok]
        if bad:
            raise RuntimeError(f"{tag} validation failed: {bad}")
        meta["validation"] = "PASSED"
        return dict(n_points=int(n), n_random_points=int(len(chk)), seed=int(ctx.seed), color_policy=policy,
                    bbox_min=bmin.tolist(), bbox_max=bmax.tolist(), confidence="constant", mean_confidence_used=mean_conf,
                    n_finite_original_points=int(finite.sum()), validation="PASSED")
    except Exception as e:
        meta["error"] = f"{type(e).__name__}: {e}"
        raise
    finally:
        _atomic_json(meta_path, meta)          # always leave a trace, FAILED or PASSED


run_all.CONFIGS["C4"]["after_init"] = use_random_points      # replaces the placeholder hook (which carried the not_ready flag)
assert not getattr(run_all.CONFIGS["C4"]["after_init"], "not_ready", False)
assert run_all.CONFIGS["C4"]["optim_pose"] is True and run_all.CONFIGS["C2"]["after_init"] is run_all.hook_none
print("Switches registered:", {k: v["after_init"].__name__ for k, v in run_all.CONFIGS.items() if k in ("C2", "C3", "C4")})

Switches registered: {'C2': 'hook_none', 'C3': 'hook_none', 'C4': 'use_random_points'}


## Section 8 - Experiment planning
Builds the exact list. Reused runs are listed; `stale` runs are blocked unless `FORCE = True`. The matched C3 baseline is checked for every pair, and the matched C2/C4 gaps relative to the existing C3 runs are repeated.

In [25]:
PLAN, SKIPPED, BLOCKED = [], [], []
for c in CONFIGS_TO_RUN:
    for (s, v) in PAIRS:
        st, det = AUDIT[(c, s, v)]
        if FORCE:
            PLAN.append((c, s, v, st != "missing"))
        elif st == "done":
            SKIPPED.append((c, s, v, "valid result reused"))
        elif st == "stale":
            BLOCKED.append((c, s, v, det))
        else:
            PLAN.append((c, s, v, False))
print(f"TO RUN ({len(PLAN)}):");   [print("  ", *p[:3], "(FORCE)" if p[3] else "") for p in PLAN]
print(f"REUSED ({len(SKIPPED)}):"); [print("  ", *p[:3]) for p in SKIPPED]
print(f"BLOCKED - stale, need FORCE=True ({len(BLOCKED)}):"); [print("  ", *p[:3], "->", p[3][:160]) for p in BLOCKED]
no_c3 = sorted({(s, v) for c in CONFIGS_TO_RUN if c != "C3" for (s, v) in PAIRS if AUDIT[("C3", s, v)][0] != "done"
                and ("C3", s, v) not in {(x[0], x[1], x[2]) for x in PLAN}})
if no_c3:
    print("WARNING - no valid matched C3 for", no_c3, "-> add 'C3' to CONFIGS_TO_RUN; these pairs cannot be compared.")
print("\nMatched gaps relative to existing C3 runs:  C2 missing:", MISSING_MATCHED_C2, "| C4 missing:", MISSING_MATCHED_C4)
if MODE == "full" and "C2" in CONFIGS_TO_RUN and not PAIRS_FROM_EXISTING_C3:
    uncovered = sorted(set(MISSING_MATCHED_C2) - set(PAIRS))
    if uncovered:
        print("NOTE - matched C2 runs NOT in this plan:", uncovered, "(set PAIRS_FROM_EXISTING_C3 = True to cover them)")
minutes = {3: 9, 6: 14, 9: 19}
est = sum(minutes[v] for _, _, v, _ in PLAN)
print(f"\nEstimated GPU time on a T4: about {est // 60} h {est % 60} min (+1-2 min per run without MASt3R cache)")

TO RUN (1):
   C4 garden 3 
REUSED (0):
BLOCKED - stale, need FORCE=True (0):

Matched gaps relative to existing C3 runs:  C2 missing: [('bonsai', 3), ('bonsai', 6), ('bonsai', 9), ('garden', 6), ('garden', 9)] | C4 missing: [('bonsai', 3), ('bonsai', 6), ('bonsai', 9), ('garden', 3), ('garden', 6), ('garden', 9)]

Estimated GPU time on a T4: about 0 h 9 min (+1-2 min per run without MASt3R cache)


## Section 9 - Dry run
Prints the exact shell commands per planned run (no GPU, nothing changed). Runs whenever `DRY_RUN = True`.

In [26]:
if DRY_RUN:
    for c, s, v, f in PLAN:
        run_all.run_one(ctx, c, s, v, force=True, dry_run=True)
    print("\nDRY RUN finished - nothing was started. Set DRY_RUN = False to execute.")
else:
    print("DRY_RUN is False - skipped.")

DRY_RUN is False - skipped.


## Execution helpers, validation and pilot gate
`validate_run` separates **critical** checks (a failure marks the run FAILED) from **warnings**. The pilot gate file `results/_pilot/pilot_C4_garden_3_seed<seed>.json` is written only when the C4 / garden / 3 views run passed every critical check; it records seed, colour policy and InstantSplat commit. `MODE = "full"` refuses to start without a matching gate.

In [27]:
def validate_run(c, s, v):
    d = Path(run_all.result_dir(ctx, c, s, v))
    out = dict(config=c, scene=s, n_views=v, checks={}, warnings=[], passed=False)
    try:
        src_dir = d if (d / "result.json").exists() else Path(run_all.result_dir(ctx, ctx.reuse.get(c, c), s, v))
        r = json.load(open(src_dir / "result.json"))
        ck = out["checks"]
        ck["result_matches_request"] = not _mismatches(r, c, s, v)
        m = {k: r.get(k) for k in ("psnr", "ssim", "lpips", "ate", "rot_deg", "rpe_deg", "ate_init", "rot_deg_init")}
        ck["metrics_finite"] = all(isinstance(x, (int, float)) and math.isfinite(x) for x in m.values())
        ck["metrics_plausible"] = ck["metrics_finite"] and 5 < m["psnr"] < 45 and 0 <= m["ssim"] <= 1 and 0 <= m["lpips"] <= 1.5
        ck["rendering_produced_files"] = r.get("n_test_images", 0) > 0 and any(p.stat().st_size > 0 for p in (src_dir / "test_renders").glob("*"))
        if r.get("n_test_images") != len(manifest[s]["test_images"]):
            out["warnings"].append(f"n_test_images {r.get('n_test_images')} != manifest test photos {len(manifest[s]['test_images'])}")
        st = r.get("stage_seconds") or {}
        ck["timings_recorded"] = all(k in st for k in ("train", "render_test", "metrics")) and r.get("peak_vram_mb") is not None
        info, pc = pose_stage_checks(c, s, v)
        out["pose_info"] = info
        ck.update(pc)
        if c == "C4":
            meta = json.load(open(d / "random_points_meta.json"))
            ck["hook_validation_passed"] = meta.get("validation") == "PASSED"
            ck["point_count_equal"] = meta.get("n_random_points") == meta.get("n_original_points") == (meta.get("preflight") or {}).get("n_vertices")
            ck["seed_matches"] = meta.get("seed") == RANDOM_SEED == r.get("seed")
            ck["color_policy_matches"] = meta.get("color_policy") == C4_COLOR_POLICY == (r.get("switch_info") or {}).get("color_policy")
            ck["confidence_constant_same_length"] = all((meta.get("checks") or {"x": False})[k] for k in ("confidence_len", "confidence_constant", "confidence_finite"))
            ck["bbox_checks"] = all((meta.get("checks") or {"x": False})[k] for k in ("inside_bbox", "coordinates_finite"))
            ck["replacement_preserved_camera_files"] = bool((meta.get("checks") or {}).get("camera_and_other_files_unchanged"))   # replacement stage
            ck["all_hook_checks"] = all((meta.get("checks") or {"x": False}).values())
            for base in ("C3", "C2"):                                                        # same MASt3R init -> same starting poses
                bd = Path(run_all.result_dir(ctx, base, s, v)) if base != "C3" else Path(run_all.result_dir(ctx, "C3_baseline", s, v))
                if (bd / "poses/pose_org.npy").exists() and (d / "poses/pose_org.npy").exists():
                    diff = float(np.abs(np.load(bd / "poses/pose_org.npy") - np.load(d / "poses/pose_org.npy")).max())
                    if diff > 1e-3:
                        out["warnings"].append(f"starting poses differ from {base} by {diff:.2e} (fresh MASt3R inference is not guaranteed bit-identical)")
                    out["starting_pose_diff_vs_" + base] = diff
        out["passed"] = all(ck.values())
    except Exception as e:
        out["error"] = f"{type(e).__name__}: {e} [{c} {s} {v} views, stage: validation]"
    return out


def _tar_hashes(pairs):
    return {f"{s}/{v}": _sha(f"{run_all.init_cache_dir(ctx, s, v)}/sparse.tar")
            for s, v in pairs if os.path.exists(f"{run_all.init_cache_dir(ctx, s, v)}/sparse.tar")}


def execute_plan(plan, label):
    t0 = time.time()
    pairs = sorted({(s, v) for _, s, v, _ in plan})
    h_before = _tar_hashes(pairs)
    log = dict(label=label, session=SESSION, started=datetime.datetime.now().isoformat(timespec="seconds"), gpu=GPU_NAME,
               commit=COMMIT, seed=RANDOM_SEED, color_policy=C4_COLOR_POLICY, runs=[], not_executed=[])
    stopped = False
    for i, (c, s, v, f) in enumerate(plan):
        d = Path(run_all.result_dir(ctx, c, s, v))
        if f and (d / "result.json").exists():
            shutil.copy(d / "result.json", d / f"result.superseded_{SESSION}.json")      # never lose the previous result
        done, failed = run_all.run_batch(ctx, [c], [s], [v], force=f)
        entry = dict(config=c, scene=s, n_views=v, executed=bool(done), run_error=(failed[0][3] if failed else None))
        if done:
            val = validate_run(c, s, v)
            _atomic_json(d / "validation.json", val)
            entry["validation"] = val
            if not val["passed"]:
                print(f"   VALIDATION FAILED for {c}/{s}/{v}: {val.get('error') or [k for k, ok in val['checks'].items() if not ok]}")
                os.replace(d / "result.json", d / "result.FAILED.json")        # excluded from aggregation, logs kept
                entry["marked_failed"] = True
            else:
                print(f"   validation PASSED for {c}/{s}/{v}" + (f" (warnings: {val['warnings']})" if val["warnings"] else ""))
        log["runs"].append(entry)
        if STOP_ON_FAILURE and not (entry["executed"] and entry["validation"]["passed"]):
            log["not_executed"] = [list(x[:3]) for x in plan[i + 1:]]
            print(f"\nSTOPPED after the first failure; {len(log['not_executed'])} planned run(s) were NOT started.")
            stopped = True
            break
    h_after = _tar_hashes(pairs)
    for k, h in h_before.items():
        if h_after.get(k) != h:
            raise RuntimeError(f"CRITICAL: MASt3R init cache {k} was modified during the batch!")
    log.update(init_cache_unchanged=True, stopped_early=stopped, minutes=round((time.time() - t0) / 60, 1))
    _atomic_json(f"{DRIVE}/logs/S2_S3_{label}_{SESSION}.json", log)
    print(f"\nInit caches unchanged ({len(h_before)} checked). Log: Drive/logs/S2_S3_{label}_{SESSION}.json")
    return log


def gate_path():
    return f"{RESULTS}/_pilot/pilot_C4_garden_3_seed{RANDOM_SEED}.json"


def write_pilot_gate():
    # re-validates the pilot run from disk; the gate exists only if every critical check passed
    os.makedirs(f"{RESULTS}/_pilot", exist_ok=True)
    st = classify("C4", "garden", 3)[0]
    val = validate_run("C4", "garden", 3) if st == "done" else dict(passed=False, error=f"pilot run status is {st}")
    gate = dict(passed=bool(val["passed"]), seed=RANDOM_SEED, color_policy=C4_COLOR_POLICY, commit=COMMIT, gpu=GPU_NAME,
                date=datetime.datetime.now().isoformat(timespec="seconds"), validation=val)
    if gate["passed"]:
        _atomic_json(gate_path(), gate)
    elif os.path.exists(gate_path()):
        os.replace(gate_path(), gate_path() + f".invalidated_{SESSION}")
    return gate


def pilot_gate_ok():
    if not os.path.exists(gate_path()):
        return False, "no pilot gate file"
    g = json.load(open(gate_path()))
    ok = g.get("passed") and g.get("seed") == RANDOM_SEED and g.get("color_policy") == C4_COLOR_POLICY and (DRY_RUN or g.get("commit") == COMMIT)
    return bool(ok), ("gate valid" if ok else f"gate does not match this session (seed/policy/commit): {g.get('seed')}, {g.get('color_policy')}, {g.get('commit')}")

print("execution helpers ready")

execution helpers ready


## Section 10 - Pilot execution (strict)
Runs only when `MODE = "pilot"` and `DRY_RUN = False`: **C4 / garden / 3 views / fixed seed**. It validates the real initialization (preflight), point count, bounding box, confidence array, camera files, training command, rendering and the evaluation metrics, then writes the pilot gate. If anything critical fails, the gate is not written, the failure banner is printed and the full matrix stays blocked.

In [28]:
EXEC_LOG, PILOT_GATE = None, None
if MODE == "pilot" and not DRY_RUN:
    EXEC_LOG = execute_plan(PLAN, "pilot")
    PILOT_GATE = write_pilot_gate()
    if PILOT_GATE["passed"]:
        print("\nPILOT PASSED - gate written:", gate_path(), "\nNow set MODE = 'full' (same seed and colour policy) to continue.")
    else:
        print("\n" + "!" * 70 + "\nPILOT FAILED - the full matrix stays blocked. Details:\n", json.dumps(PILOT_GATE["validation"], indent=1, default=str)[:3000], "\n" + "!" * 70)
else:
    print("Pilot skipped (needs MODE = 'pilot' and DRY_RUN = False).")

=== C4/garden/3_views (random points, refinement on)
   init_geo     from Drive cache (originally 68 s)
   train           77.1 s
   render_test    364.6 s
   metrics         56.0 s
   done: PSNR 15.11 | pose before->after: ATE 0.019->0.019, rot 1.09->1.76 deg | 8.5 min

Batch finished in 8.5 min: 1 runs ready, 0 not done.
   validation PASSED for C4/garden/3

Init caches unchanged (1 checked). Log: Drive/logs/S2_S3_pilot_2026-11-17_2055.json

PILOT PASSED - gate written: /content/drive/MyDrive/3dgs-project/results/_pilot/pilot_C4_garden_3_seed0.json 
Now set MODE = 'full' (same seed and colour policy) to continue.


## Section 11 - Full experiment execution
Runs the plan when `MODE = "full"` and `DRY_RUN = False`, **only if a matching pilot gate exists**. The batch stops at the first failed run or failed validation (`STOP_ON_FAILURE`); the remaining runs are listed as not started. Finished runs are reused on restart.

In [29]:
ok, why = pilot_gate_ok()
if MODE == "full":
    print("pilot gate:", why)
if MODE == "full" and not DRY_RUN:
    if not ok and not ALLOW_FULL_WITHOUT_PILOT:
        raise SystemExit("Full execution blocked: run the strict pilot first (MODE = 'pilot', DRY_RUN = False).")
    EXEC_LOG = execute_plan(PLAN, "full")
else:
    print("Full execution skipped (needs MODE = 'full' and DRY_RUN = False).")

Full execution skipped (needs MODE = 'full' and DRY_RUN = False).


## Section 12 - Result validation
Re-validates every finished run in scope (also runs from earlier sessions; nothing is renamed here): metrics finite and plausible, render files present, timings and VRAM, command vs config, **S2 poses unchanged**, **C3/C4 refinement active (poses moved)**, and for C4 the hook evidence (equal point counts, seed, colour policy, bounding box, constant confidence of the same length, camera files unchanged during replacement).

In [30]:
VALIDATION = {}
for c in CONFIGS_TO_RUN:
    for (s, v) in PAIRS:
        if classify(c, s, v)[0] == "done":
            VALIDATION[(c, s, v)] = validate_run(c, s, v)
for (c, s, v), val in VALIDATION.items():
    bad = [k for k, ok in val["checks"].items() if not ok]
    pi = val.get("pose_info", {})
    print(f"{c} {s:9s} {v} views: {'PASSED' if val['passed'] else 'FAILED'} | pose diff {pi.get('max_abs_pose_difference', float('nan')):.2e}",
          "|", val.get("error") or bad or "", "| warnings:" if val["warnings"] else "", val["warnings"] or "")
if not VALIDATION:
    print("No finished runs in scope yet.")

C4 garden    3 views: PASSED | pose diff 6.12e-03 |   


## Section 13 - Results aggregation
Backs up the current CSVs to `results/_backups/<time>/`, then regenerates `all_runs.csv`, `means_by_config.csv`, `effects.csv` from **all** valid `result.json` files with the unchanged `aggregate.py` (existing runs are rebuilt from their own result files, so nothing is lost; `result.FAILED.json` runs are excluded). Row counts are cross-checked.

In [31]:
import pandas as pd
bk = f"{RESULTS}/_backups/{SESSION}"
os.makedirs(bk, exist_ok=True)
for f in ["all_runs.csv", "means_by_config.csv", "effects.csv"]:
    if os.path.exists(f"{RESULTS}/{f}"):
        shutil.copy(f"{RESULTS}/{f}", f"{bk}/{f}")
rows, means, effects = aggregate.main(RESULTS)
df_all = pd.read_csv(f"{RESULTS}/all_runs.csv")
assert len(df_all) == len(rows) == len(aggregate.load(RESULTS)), "CSV / result.json mismatch"
assert df_all.duplicated(["config", "scene", "n_views"]).sum() == 0, "duplicate runs in all_runs.csv"
print("\nruns per config:", df_all.groupby("config").size().to_dict())
display(df_all.round(3))

8 runs -> all_runs.csv, means_by_config.csv, effects.csv in /content/drive/MyDrive/3dgs-project/results
  C2 3 photos (1 scenes): PSNR 14.62  ATE 0.019  rot 1.09 deg
  C3 3 photos (2 scenes): PSNR 14.71  ATE 0.062  rot 3.98 deg
  C3 6 photos (2 scenes): PSNR 16.54  ATE 0.052  rot 2.95 deg
  C3 9 photos (2 scenes): PSNR 17.34  ATE 0.079  rot 1.49 deg
  C4 3 photos (1 scenes): PSNR 15.11  ATE 0.019  rot 1.76 deg
  RQ2 3D points (C3 - C4) at 3 photos (1 scenes): dPSNR -0.40, positive in 0/1 scenes
  RQ3 refinement (C3 - C2) at 3 photos (1 scenes): dPSNR +0.09, positive in 1/1 scenes

runs per config: {'C2': 1, 'C3': 6, 'C4': 1}


,config,scene,n_views,psnr,ssim,lpips,ate,rot_deg,rpe_deg,ate_init,rot_deg_init,total_seconds_incl_init,total_seconds,peak_vram_mb,date
0,C2,garden,3,14.619,0.272,0.553,0.019,1.092,1.814,0.019,1.092,573.6,573.6,3241,2026-11-17T09:43:39
1,C3,bonsai,3,14.706,0.358,0.578,0.107,6.855,2.011,0.107,6.863,607.4,607.4,3261,2026-11-16T21:17:30
2,C3,bonsai,6,15.305,0.393,0.503,0.078,4.581,3.412,0.078,4.542,953.9,953.9,3695,2026-11-16T21:33:25
3,C3,bonsai,9,16.537,0.447,0.446,0.138,2.228,2.378,0.138,2.220,1272.7,1272.7,4913,2026-11-16T21:54:38
4,C3,garden,3,14.710,0.275,0.553,0.018,1.109,1.864,0.019,1.092,580.6,580.6,3241,2026-11-15T23:37:08
5,C3,garden,6,17.781,0.357,0.392,0.026,1.320,1.683,0.026,1.332,814.1,814.1,3653,2026-11-15T23:50:43
6,C3,garden,9,18.151,0.352,0.378,0.019,0.750,0.985,0.020,0.754,1020.3,1020.3,4827,2026-11-16T21:07:22
7,C4,garden,3,15.105,0.246,0.701,0.019,1.763,1.944,0.019,1.092,575.2,507.6,1701,2026-11-17T21:03:37


## Section 14 - Comparison and ablation analysis
A variant run (C2 or C4) is compared with C3 **only if scene and number of views match exactly and every experimental parameter matches** (iterations, resolution, test-pose iterations, stride, InstantSplat commit, training photo list). Pairs that fail are listed and excluded; variant runs without a C3 partner and C3 runs without a partner are listed too. No scene averages over unmatched sets are formed here. The tables from `aggregate.py` above (`effects.csv`) compare scenes present in both configs but do not check run parameters, so treat them as secondary. No significance is claimed: look at `n_pairs` and `scenes_positive`; differences below ~0.3 dB PSNR are within run-to-run noise.

In [32]:
PARAM_KEYS = ["iterations", "resolution", "optim_test_pose_iter", "test_stride", "instantsplat_commit", "train_images"]
R = {(r["config"], r["scene"], r["n_views"]): r for r in aggregate.load(RESULTS)}
matched, excluded, unmatched = [], [], []
for (cfg, s, v), r in sorted(R.items()):
    if cfg not in ("C2", "C4"):
        continue
    base = R.get(("C3", s, v))
    if base is None:
        unmatched.append(f"{cfg} {s} {v} views: no C3 run with the same scene and views")
        continue
    diff = [k for k in PARAM_KEYS if r.get(k) != base.get(k)]
    if diff:
        excluded.append(f"{cfg} {s} {v} views: parameters differ from C3 in {diff}")
        continue
    if r.get("gpu") != base.get("gpu"):
        print(f"note: {cfg} {s} {v}: GPU differs from C3 ({r.get('gpu')} vs {base.get('gpu')})")
    row = dict(comparison="S2: refinement OFF (C2 - C3)" if cfg == "C2" else "S3: random points (C4 - C3)", scene=s, n_views=v)
    for m in ("psnr", "ssim", "lpips", "ate", "rot_deg", "rpe_deg"):
        row[f"d_{m}"] = r[m] - base[m]
    matched.append(row)
for (cfg, s, v) in sorted(R):
    if cfg == "C3" and not any(("C2", s, v) in R or ("C4", s, v) in R for _ in [0]):
        unmatched.append(f"C3 {s} {v} views: no C2 or C4 partner yet")
print("EXCLUDED (parameter mismatch):", excluded or "none")
print("UNMATCHED:", *unmatched, sep="\n  ") if unmatched else print("UNMATCHED: none")
mdf = pd.DataFrame(matched)
if len(mdf):
    display(mdf.round(3))
    summ = mdf.groupby(["comparison", "n_views"]).agg(n_pairs=("scene", "count"), d_psnr=("d_psnr", "mean"), d_ssim=("d_ssim", "mean"),
                                                       d_lpips=("d_lpips", "mean"), d_ate=("d_ate", "mean"), d_rot_deg=("d_rot_deg", "mean"),
                                                       scenes_positive=("d_psnr", lambda x: f"{int((x > 0).sum())}/{len(x)}")).round(3)
    display(summ)
    print("ATE is comparable only between runs with the same number of photos (grouped above).")
    print("C2 differs from C3 only by --optim_pose. C4 differs from C3 only by the replaced points (positions + constant confidence; colour policy:", C4_COLOR_POLICY + ").")
else:
    print("No matched C3 vs C2/C4 pairs yet.")
display(pd.read_csv(f"{RESULTS}/effects.csv").round(3) if os.path.exists(f"{RESULTS}/effects.csv") else "no effects.csv")

EXCLUDED (parameter mismatch): none
UNMATCHED:
  C3 bonsai 3 views: no C2 or C4 partner yet
  C3 bonsai 6 views: no C2 or C4 partner yet
  C3 bonsai 9 views: no C2 or C4 partner yet
  C3 garden 6 views: no C2 or C4 partner yet
  C3 garden 9 views: no C2 or C4 partner yet


,comparison,scene,n_views,d_psnr,d_ssim,d_lpips,d_ate,d_rot_deg,d_rpe_deg
0,S2: refinement OFF (C2 - C3),garden,3,-0.091,-0.003,-0.000,0.001,-0.017,-0.051
1,S3: random points (C4 - C3),garden,3,0.395,-0.029,0.147,0.001,0.654,0.079


,,n_pairs,d_psnr,d_ssim,d_lpips,d_ate,d_rot_deg,scenes_positive
comparison,n_views,,,,,,,
S2: refinement OFF (C2 - C3),3,1,-0.091,-0.003,-0.000,0.001,-0.017,0/1
S3: random points (C4 - C3),3,1,0.395,-0.029,0.147,0.001,0.654,1/1


ATE is comparable only between runs with the same number of photos (grouped above).
C2 differs from C3 only by --optim_pose. C4 differs from C3 only by the replaced points (positions + constant confidence; colour policy: preserve_original).


,effect,n_views,n_scenes,scenes,d_psnr,sd_psnr,scenes_positive_psnr,d_ssim,d_lpips,d_rot_deg,d_rpe_deg,d_ate
0,RQ2 3D points (C3 - C4),3,1,garden,-0.395,NaN,0/1,0.029,-0.147,-0.654,-0.079,-0.001
1,RQ3 refinement (C3 - C2),3,1,garden,0.091,NaN,1/1,0.003,0.000,0.017,0.051,-0.001


## Section 15 - Export and final audit
Writes `results/audit_S2_S3_<time>.md` (settings, versions, plan, what ran, validation, pilot gate, init-cache protection) and prints it. A run counts as successful only if its validation passed.

In [33]:
import torch
ver = dict(python=platform.python_version(), numpy=np.__version__, torch=torch.__version__, cuda=torch.version.cuda, gpu=GPU_NAME,
           instantsplat=COMMIT, pose_error_version=pose_error.VERSION)
exec_runs = (EXEC_LOG or {}).get("runs", [])
ok_runs = [e for e in exec_runs if e.get("validation", {}).get("passed")]
bad_runs = [e for e in exec_runs if not e.get("validation", {}).get("passed")]
gate_ok, gate_msg = pilot_gate_ok()
lines = [f"# S2/S3 audit - session {SESSION}", "",
         f"- settings: MODE={MODE}, CONFIGS_TO_RUN={CONFIGS_TO_RUN}, PAIRS={PAIRS}, RANDOM_SEED={RANDOM_SEED}, C4_COLOR_POLICY={C4_COLOR_POLICY}, DRY_RUN={DRY_RUN}, FORCE={FORCE}",
         f"- versions: {ver}", f"- planned: {len(PLAN)}, reused: {len(SKIPPED)}, blocked (stale): {len(BLOCKED)}",
         f"- executed this session: {len(exec_runs)} (validated OK: {len(ok_runs)}, failed/not validated: {len(bad_runs)}), not started after stop: {(EXEC_LOG or {}).get('not_executed', [])}",
         f"- pilot gate: {gate_msg}",
         f"- matched gaps vs existing C3 -> C2 missing: {MISSING_MATCHED_C2}; C4 missing: {MISSING_MATCHED_C4}",
         f"- MASt3R init caches unchanged: {(EXEC_LOG or {}).get('init_cache_unchanged', 'n/a (nothing executed)')}", "", "## Validation of runs in scope"]
for (c, s, v), val in VALIDATION.items():
    lines.append(f"- {c} {s} {v} views: {'PASSED' if val['passed'] else 'FAILED'} {json.dumps(val.get('pose_info', {}), default=str)} warnings={val['warnings']}")
for b in BLOCKED:
    lines.append(f"- BLOCKED {b[0]} {b[1]} {b[2]} views: {b[3]}")
for e in bad_runs:
    lines.append(f"- NOT SUCCESSFUL {e['config']} {e['scene']} {e['n_views']}: {e.get('run_error') or e.get('validation', {}).get('error') or 'validation failed'}")
audit_path = f"{RESULTS}/audit_S2_S3_{SESSION}.md"
open(audit_path + ".tmp", "w").write("\n".join(lines) + "\n"); os.replace(audit_path + ".tmp", audit_path)
print("\n".join(lines)); print("\nSaved:", audit_path)
print("CSV/JSON files:", RESULTS, "| per-run folders: results/<config>/<scene>/<n>_views/ (result.json, random_points_meta.json, validation.json, logs, poses/)")

# S2/S3 audit - session 2026-11-17_2055

- settings: MODE=pilot, CONFIGS_TO_RUN=['C4'], PAIRS=[('garden', 3)], RANDOM_SEED=0, C4_COLOR_POLICY=preserve_original, DRY_RUN=False, FORCE=False
- versions: {'python': '3.13.16', 'numpy': '2.4.6', 'torch': '2.14.1+cu130', 'cuda': '13.0', 'gpu': 'Tesla T4, 15360 MiB', 'instantsplat': 'b951567', 'pose_error_version': 2}
- planned: 1, reused: 0, blocked (stale): 0
- executed this session: 1 (validated OK: 1, failed/not validated: 0), not started after stop: []
- pilot gate: gate valid
- matched gaps vs existing C3 -> C2 missing: [('bonsai', 3), ('bonsai', 6), ('bonsai', 9), ('garden', 6), ('garden', 9)]; C4 missing: [('bonsai', 3), ('bonsai', 6), ('bonsai', 9), ('garden', 3), ('garden', 6), ('garden', 9)]
- MASt3R init caches unchanged: True

## Validation of runs in scope
- C4 garden 3 views: PASSED {"max_abs_pose_difference": 0.006119847297668457, "abs_ate_change": 8.469648333962743e-05, "abs_rot_change_deg": 0.6710427871498803} warnings=[]

Sa